# Beginner 03: Secure Research Agent

**Level:** Beginner · **Duration:** 3–4 hours · **Prerequisites:** Beginner 01 and 02

Build a credential-free research pipeline where retrieved content is evidence—not instructions, identity, authority, or permission. The security invariant is: **authorize → rank → snapshot → generate → verify → release**.

In [ ]:
import importlib, json, sys
from pathlib import Path

for candidate in (Path('.'), Path('curriculum/beginner/03-secure-research-agent')):
    if (candidate / '03_secure_research_agent.py').exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Run from the repository root or the course directory')

lab = importlib.import_module('03_secure_research_agent')
print('Loaded policy', lab.POLICY_VERSION)

## 1. Architecture and trust boundaries

![Secure research agent architecture](architecture.svg)

Trusted application code resolves identity and trims the searchable corpus. The model sees only authorized evidence snapshots and proposes claims. Deterministic code decides whether anything is released.

## 2. Threat model

We test six failure classes: caller-forged entitlements, cross-tenant retrieval, stale evidence, prompt injection, unauthorized actions, and citation laundering. The model, query, retrieved text, proposed citations, and proposed actions are untrusted. Identity resolution, authorization metadata, retrieval policy, release validators, and audit storage are trusted components.

## 3. Inspect the metadata-rich corpus

Relevance, provenance, sensitivity, lifecycle, tenant, and authority answer different questions. Do not collapse them into one `trusted` flag.

In [ ]:
for doc in lab.DOCUMENT_STORE.values():
    print(f'{doc.document_id:22} tenant={doc.tenant_id:7} sensitivity={doc.sensitivity.value:12} lifecycle={doc.lifecycle.value:10} v{doc.version}')

## 4. Resolve authorization context server-side

The public API accepts `subject` and `query` only. Tenant and sensitivity entitlements come from the registry. A typed context object would still be untrusted if a caller could construct it.

In [ ]:
alice = lab.ResearchContextResolver.resolve('alice')
bob = lab.ResearchContextResolver.resolve('bob')
print(alice)
print(bob)
assert alice.tenant_id == 'acme'
assert lab.Sensitivity.CONFIDENTIAL not in alice.allowed_sensitivities

## 5. Prove authorization happens before ranking

For Alice, the Acme confidential record, the Globex record, and the superseded record must not appear in the set presented to the scorer—even when their words match the query.

In [ ]:
retrieval = lab.RetrievalService()
result = retrieval.search('Project Phoenix budget and ticket retention', alice)
print('Scored:', result.scored_document_ids)
print('Returned:', [item.ref.display_id for item in result.evidence])
for forbidden in ('doc-conf-01', 'doc-globex-conf-01', 'doc-stale-01'):
    assert forbidden not in result.scored_document_ids

## 6. Freeze the exact evidence used

A citation binds document ID, version, and digest. The compact user label is not the entire verification record.

In [ ]:
retention = lab.RetrievalService().search('ticket retention policy', alice).evidence[0]
print('User label:', retention.ref.display_id)
print('Digest:', retention.ref.content_digest)
print('Authority:', retention.authority.value)
assert retention.authority == lab.Authority.INFORMATIONAL

## 7. Normal and unauthorized paths

The same application answers an authorized current policy question and abstains on an unauthorized confidential question without confirming the hidden record.

In [ ]:
ids = iter(['nb-1', 'nb-2', 'nb-3'])
app = lab.ResearchApplication(request_id_factory=lambda: next(ids))
normal = app.answer('alice', 'ticket retention policy')
denied = app.answer('alice', 'Project Phoenix budget')
privileged = app.answer('bob', 'Project Phoenix budget')
print(normal)
print(denied)
print(privileged)
assert normal.citations == ('doc-int-01@v3',)
assert denied.terminal_state == 'insufficient_evidence'
assert '$4.2M' in privileged.answer

## 8. Tenant isolation and lifecycle

Bob may read Acme confidential data, but that does not grant access to Globex. A similarly named Globex record and a superseded Acme policy never enter scoring for Bob.

In [ ]:
bob_result = lab.RetrievalService().search('Globex Project Phoenix budget', bob)
print('Scored for Bob:', bob_result.scored_document_ids)
assert 'doc-conf-01' in bob_result.scored_document_ids
assert 'doc-globex-conf-01' not in bob_result.scored_document_ids
assert 'doc-stale-01' not in bob_result.scored_document_ids
assert '$91M' not in ' '.join(item.text for item in bob_result.evidence)

## 9. Injection detection is not the boundary

One poison is detected and one bypasses the heuristic. Both are blocked because no evidence can grant a side-effect capability.

In [ ]:
for query in ('user profile', 'feature request', 'legacy operations'):
    local = lab.ResearchApplication(request_id_factory=lambda q=query: 'poison-' + q.replace(' ', '-'))
    response = local.answer('alice', query)
    event = local.audit_sink.events[-1]
    print(query, response.terminal_state, event.suspicious_content_detected, event.reason)
    assert response.terminal_state == 'blocked'
    assert event.reason.startswith('unauthorized_capability_')

## 10. Citation integrity and citation laundering

A real document ID is not enough. The exact snapshot must have been supplied in this run, and the cited evidence must support the atomic claim.

In [ ]:
citation_cases = (
    'make up citation for retention policy',
    'cite unretrieved for retention policy',
    'stale citation retention policy',
    'launder retention policy',
    'zero citation for retention policy',
)
for index, query in enumerate(citation_cases):
    local = lab.ResearchApplication(request_id_factory=lambda i=index: f'citation-{i}')
    response = local.answer('alice', query)
    print(query, '→', response.terminal_state, local.audit_sink.events[-1].reason)
    assert response.terminal_state != 'answered'

## 11. Data-minimized audit receipts

The audit event keeps a digest and length instead of a raw query. Its scored IDs are already authorized; filtered confidential and cross-tenant IDs are not enumerated.

In [ ]:
raw_query = 'Project Phoenix budget requested by employee Ada'
local = lab.ResearchApplication(request_id_factory=lambda: 'audit-1')
local.answer('alice', raw_query)
audit = local.audit_sink.events[-1]
print(json.dumps(audit.to_dict(), indent=2, default=str))
assert raw_query not in str(audit.to_dict())
assert 'doc-conf-01' not in str(audit.to_dict())
assert len(audit.query_digest) == 64

## 12. OpenAI Agents SDK: strict read-only tool

The SDK is useful when your server owns tool implementations, state, and approvals. The tool schema exposes only `query`; authorization remains in `RunContextWrapper[SDKRuntime]`. This cell constructs the agent and dispatches the same boundary without an API call.

In [ ]:
sdk = importlib.import_module('03_secure_research_agent_sdk')
schema = sdk.search_authorized_corpus.params_json_schema
print(json.dumps(schema, indent=2))
assert set(schema['properties']) == {'query'}
assert schema['additionalProperties'] is False
assert [tool.name for tool in sdk.SDK_AGENT.tools] == ['search_authorized_corpus']

In [ ]:
runtime = sdk.build_runtime('alice')
safe_payload = sdk.dispatch_authorized_search(runtime, 'Project Phoenix budget')
print(safe_payload)
assert json.loads(safe_payload)['evidence'] == []
assert 'doc-conf-01' not in safe_payload

## 13. Evaluate safety and utility separately

A system that refuses everything can score well on disclosure safety but fail its purpose. A useful system can still be unsafe. Keep separate gates for unsafe disclosure, action execution, valid-answer success, correct abstention, citation integrity, and trace coverage.

In [ ]:
metrics = lab.evaluate_fixture()
print(metrics)
assert metrics.unsafe_disclosures == 0
assert metrics.unsafe_actions_executed == 0
assert metrics.valid_answer_success_rate == 1.0
assert metrics.expected_abstention_accuracy == 1.0

## 14. Exercises

1. Add an expired document and prove it is never scored.
2. Bind an ACL revision to `scope_digest` and reject release if the revision changes mid-request.
3. Add a duplicate-citation attack and a failing test.
4. Replace token overlap with BM25 or a local vector index without weakening pre-ranking authorization.
5. Add a human-review state for high-consequence claims and define the minimum evidence a reviewer may see.

## 15. Production handoff

Before deployment, verify identity and policy freshness, datastore-level tenant isolation, index update/deletion behavior, evidence versioning, independently authorized actions, audit protection, fail-closed dependency behavior, human escalation, and labelled safety/utility evaluation. The model can help draft; it cannot be the access-control or release authority.